# Data Exploration
Explore the GuitarSet dataset: inspect JAMS annotations, verify audio files, and identify monophonic segments suitable for training.

In [ ]:
import jams
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from pathlib import Path

In [ ]:
ANNOTATION_DIR = Path('../data/guitarset/annotations')
AUDIO_DIR = Path('../data/guitarset/audio')

annotation_files = sorted(ANNOTATION_DIR.glob('*.jams'))
print(f'Found {len(annotation_files)} annotation files')

## Inspect a single JAMS file

In [ ]:
jam = jams.load(str(annotation_files[0]))
print(jam)

## Check note annotations per string

In [ ]:
note_data = jam.search(namespace='note_midi')
print(f'Found {len(note_data)} string tracks')
for i, track in enumerate(note_data):
    print(f'String {i}: {len(track.data)} notes')

## Visualize note activity across strings

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
for i, track in enumerate(note_data):
    for _, row in track.data.iterrows():
        ax.barh(i, row.duration, left=row.time.total_seconds(), height=0.6)
ax.set_xlabel('Time (s)')
ax.set_ylabel('String')
ax.set_title(annotation_files[0].stem)
plt.tight_layout()
plt.show()

## Identify monophonic segments

In [ ]:
def count_simultaneous_strings(jam, resolution=0.01):
    """Return a time series of how many strings are active at each timestep."""
    note_data = jam.search(namespace='note_midi')
    duration = jam.file_metadata.duration
    times = np.arange(0, duration, resolution)
    active = np.zeros(len(times), dtype=int)
    for track in note_data:
        for _, row in track.data.iterrows():
            onset = row.time.total_seconds()
            offset = onset + row.duration
            mask = (times >= onset) & (times < offset)
            active[mask] += 1
    return times, active

times, active = count_simultaneous_strings(jam)
mono_ratio = np.mean(active <= 1)
print(f'Monophonic (0-1 strings active): {mono_ratio:.1%} of the file')

## Load and inspect the corresponding audio

In [ ]:
audio_file = AUDIO_DIR / (annotation_files[0].stem + '_mix.wav')
y, sr = librosa.load(str(audio_file), sr=None, mono=True)
print(f'Sample rate: {sr} Hz  |  Duration: {len(y)/sr:.1f}s  |  Samples: {len(y)}')

fig, ax = plt.subplots(figsize=(14, 3))
librosa.display.waveshow(y, sr=sr, ax=ax)
ax.set_title(audio_file.stem)
plt.tight_layout()
plt.show()